# CGT Training Methodology

The Conditional Graph Transformer (CGT) (Chen et al., 2023) generates synthetic graph data by learning to autoregressively predict sequences of discretised node features, conditioned on node labels. Rather than operating on the full graph, CGT decomposes it into local **computation graphs** — rooted ego-networks that capture each node's neighbourhood structure — and models them as token sequences using a transformer.

**High-level pipeline:**

1. **Feature quantization** — K-means clustering maps continuous node features to discrete cluster IDs (tokens)
2. **Computation graph construction** — For each node, sample a fixed-depth, fixed-fanout neighbourhood tree
3. **Sequence flattening** — Decompose each tree into root-to-leaf paths (short sequences) for the transformer
4. **XLNet training** — Train a label-conditioned XLNet to predict next tokens in these sequences via cross-entropy
5. **Autoregressive generation** — Sample new cluster ID sequences from the trained model
6. **Reconstruction** — Map generated cluster IDs back to feature vectors via cluster centres; construct fixed-structure adjacency

**Overview:**
1. Feature quantization via k-means
2. Computation graph representation
3. XLNet architecture and label conditioning
4. Training procedure
5. Autoregressive generation
6. Output format and reconstruction
7. Key design decisions
8. Usage

## 1. Feature Quantization via K-Means

Transformers operate on discrete token vocabularies. CGT bridges the gap between continuous node features and discrete sequence modelling by quantizing features into cluster IDs using constrained k-means.

**The clustering computation** (implemented in `CGT/generator/cluster.py`):

1. **PCA** — reduce features to min(feat_dim, 128) dimensions to stabilise clustering
2. **Constrained k-means** — fit `cluster_num` (default 512) clusters with minimum cluster size `cluster_size` (default 1, the k-anonymity floor) using `KMeansConstrained`
3. **Assign all nodes** — every node (not just the fit set) gets a k-anonymity-preserving cluster ID; the returned centres are then L2-normalised in the original feature space
4. **Append special tokens** — `empty_id = cluster_num` (padding for nodes with fewer neighbours than `cg_fanout`), `start_id = cluster_num + 1` (sequence start token)

The resulting vocabulary has size `cluster_num + 2`.

**Precomputed clustering cache (training).** CGT training no longer fits k-means on every run. The partition is precomputed once per `(dataset, task, trial, cluster_size, cluster_num)` by `scripts/cluster/precompute_clusters.py` and written under `cache/clustering/`. During training, `train_and_generate` (`CGT/generator/gpt/gpt.py`) calls `load_cached_clusters` (`CGT/generator/cluster.py`) to read the cached `cluster_ids` and L2-normed `l2_centers` instead of clustering on the fly, re-adding the `empty_id` padding so the rest of the pipeline keeps its `(N+1,)` / `(cluster_num+1, feat_dim)` shape contract. This serves two goals: (a) it externalises the k-anonymity partition so CGT and BiGG can be compared on the *identical* partition (the fairness lock), and (b) it avoids redundant clustering across runs. The loader **fails loud** if the cache entry is missing (no `DONE`) — it never silently re-clusters. The cache root is selected with `--cache_root` (default `cache/clustering`).

Because the producer always fits on the full fit-set, `--cluster_sample_num` no longer affects the training partition — the subsample-up-to-`cluster_sample_num` step that previously bounded k-means cost is now done once, on every fit-set node, by the producer. `cluster_sample_num` still appears in the synthetic output variant directory name, but is inert for the clustering itself.

**Why k-means?** It converts the continuous feature space into a finite vocabulary while preserving the dominant modes of the feature distribution. At reconstruction time, generated cluster IDs are mapped back to cluster centre vectors, so the quality of the synthetic features depends on how well the cluster centres represent the original distribution. Increasing `cluster_num` improves fidelity but increases the vocabulary size the transformer must model.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Simplified illustration of the quantization pipeline
np.random.seed(42)

# Simulate 200 nodes with 64-dimensional features (3 natural clusters)
n_nodes = 200
feat_dim = 64
centers_true = np.random.randn(3, feat_dim) * 3
labels_true = np.random.choice(3, n_nodes)
feats = centers_true[labels_true] + np.random.randn(n_nodes, feat_dim) * 0.5

# Step 1: PCA to lower dimension
pca = PCA(n_components=min(feat_dim, 128))
feats_pca = pca.fit_transform(feats)

# Step 2: K-means clustering (using standard KMeans for illustration)
cluster_num = 8
kmeans = KMeans(n_clusters=cluster_num, n_init=1, max_iter=8, random_state=0)
kmeans.fit(feats_pca)
cluster_centers_pca = kmeans.cluster_centers_
cluster_centers = pca.inverse_transform(cluster_centers_pca)

# Step 3: Assign each node to nearest cluster
cluster_ids = ((feats[:, None, :] - cluster_centers[None, :, :]) ** 2).sum(-1).argmin(1)

# Step 4: Reconstruct features from cluster IDs
reconstructed = cluster_centers[cluster_ids]

# Measure reconstruction error
mse = ((feats - reconstructed) ** 2).mean()

print(f"Nodes: {n_nodes}, Feature dim: {feat_dim}")
print(f"Clusters: {cluster_num}")
print(f"Vocab size: {cluster_num} + 2 (empty + start) = {cluster_num + 2}")
print(f"\nCluster assignments (first 20 nodes): {cluster_ids[:20]}")
print(f"Reconstruction MSE: {mse:.4f}")
print(f"\nOriginal feature[0][:8]:      {feats[0][:8].round(3)}")
print(f"Reconstructed feature[0][:8]: {reconstructed[0][:8].round(3)}")

## 2. Computation Graph Representation

A computation graph is a rooted tree that captures a node's local neighbourhood — the same structure a GNN aggregates over during message passing. CGT samples these trees and flattens them into token sequences for the transformer.

**Construction** (implemented in `CGT/generator/gpt/dataset.py`, `Dataset.__getitem__`):

Given a target node, expand its neighbourhood for `cg_depth` hops, sampling exactly `cg_fanout` neighbours at each hop. If a node has fewer neighbours than `cg_fanout`, pad with `empty_id`. The result is a tree of cluster IDs:

```
cg_depth=2, cg_fanout=2:

                  [start]
                    |
                  [root]              ← target node's cluster ID
                 /      \
            [n1_1]     [n1_2]         ← 1-hop neighbours' cluster IDs
            /    \     /    \
        [n2_1] [n2_2] [n2_3] [n2_4]  ← 2-hop neighbours' cluster IDs
```

**Flattening into short sequences:**

The full tree has $1 + \text{fanout} + \text{fanout}^2 + \dots$ nodes, which grows exponentially. Instead of feeding the entire tree as one sequence, CGT decomposes it into $\text{fanout}^{\text{depth}}$ **short sequences** — one per root-to-leaf path:

```
Short sequence 1: [start, root, n1_1, n2_1]   (block_size = 1 + 1 + depth = 4)
Short sequence 2: [start, root, n1_1, n2_2]
Short sequence 3: [start, root, n1_2, n2_3]
Short sequence 4: [start, root, n1_2, n2_4]
```

Each short sequence has length `block_size = 1 + 1 + cg_depth`. The transformer is trained to predict each token from the preceding tokens in its short sequence. This decomposition keeps sequence length constant regardless of graph size, making the approach scalable.

In [ ]:
import numpy as np

# Illustrate computation graph construction and flattening

cg_depth = 2
cg_fanout = 3
cluster_num = 8
empty_id = cluster_num
start_id = cluster_num + 1

# Simulate a small graph: 10 nodes, each with some neighbours
np.random.seed(7)
adjacency = {
    0: [1, 2, 3, 4],
    1: [0, 5],
    2: [0, 3, 6, 7],
    3: [0, 2, 8],
    4: [0, 9],
    5: [1],
    6: [2],
    7: [2, 8],
    8: [3, 7],
    9: [4],
}
node_cluster_ids = np.random.randint(0, cluster_num, size=10)

# Build computation graph for node 0
target = 0
tree_tokens = [start_id, node_cluster_ids[target]]
curr_targets = [target]

print(f"Building computation graph for node {target} (cluster ID={node_cluster_ids[target]})")
print(f"  cg_depth={cg_depth}, cg_fanout={cg_fanout}\n")

for hop in range(cg_depth):
    new_targets = []
    for t in curr_targets:
        neighbours = adjacency.get(t, [])
        if len(neighbours) >= cg_fanout:
            sampled = list(np.random.choice(neighbours, cg_fanout, replace=False))
        elif len(neighbours) == 0:
            sampled = [None] * cg_fanout  # will become empty_id
        else:
            sampled = neighbours + [None] * (cg_fanout - len(neighbours))

        for s in sampled:
            if s is None:
                tree_tokens.append(empty_id)
                new_targets.append(None)
            else:
                tree_tokens.append(node_cluster_ids[s])
                new_targets.append(s)
    curr_targets = new_targets
    print(f"  Hop {hop+1}: sampled {len(new_targets)} nodes")

print(f"\nFull tree tokens ({len(tree_tokens)} total): {tree_tokens}")

# Flatten into short sequences (root-to-leaf paths)
block_size = 1 + 1 + cg_depth
short_seq_num = cg_fanout ** cg_depth

print(f"\nblock_size = {block_size}")
print(f"short_seq_num = {cg_fanout}^{cg_depth} = {short_seq_num}")
print(f"\nShort sequences (each is a root-to-leaf path):")

# Reconstruct paths through the tree
for i in range(short_seq_num):
    seq = [tree_tokens[0], tree_tokens[1]]  # start, root
    idx = i
    offset = 2
    for hop in range(cg_depth):
        level_size = cg_fanout ** (hop + 1)
        # Navigate to the correct position in this level
        parent_idx = idx // (cg_fanout ** (cg_depth - hop - 1))
        seq.append(tree_tokens[offset + parent_idx])
        offset += level_size if hop < cg_depth - 1 else 0
        break
    # Simplified: just show first two levels for clarity
    parent = i // cg_fanout
    leaf_pos = 2 + cg_fanout + i
    seq = [tree_tokens[0], tree_tokens[1], tree_tokens[2 + parent], tree_tokens[leaf_pos]]
    print(f"  Seq {i}: {seq}  →  query={seq[:-1]}, target={seq[1:]}")

## 3. XLNet Architecture and Label Conditioning

CGT uses an XLNet-based transformer (Yang et al., 2019), adapted from Karpathy's minGPT. The key architectural choice is XLNet's **dual-stream self-attention**, which provides two parallel representation streams:

- **Content stream** — standard causal self-attention over the input token embeddings. Each position attends to all previous positions and itself. This builds contextual representations of the observed sequence.
- **Query stream** — a separate learnable stream that attends to the content stream's representations via cross-attention, but uses its own set of query embeddings. This stream produces the prediction logits.

The dual-stream design allows the model to use rich bidirectional-style context (via the content stream) while still producing autoregressive predictions (via the query stream). In standard GPT, the prediction at position $t$ can only use information from positions $< t$. The query stream in XLNet additionally incorporates the target position's context without seeing the target token itself.

**Label conditioning:** Node labels are embedded via a class embedding layer and added to the query stream's positional embeddings:

```python
q = drop(query_emb + position_emb[:, 1:t+1] + class_emb(labels))
```

This conditions generation on the node's class — the model learns class-specific feature distributions, which is critical for producing synthetic data that preserves label-feature correlations for downstream tasks.

**Output head:** The linear head projects to `vocab_size - 1` logits, excluding the start token from the prediction space (the model should never predict a start token mid-sequence). The start token's logit position is additionally set to $-\infty$ to prevent sampling it.

In [ ]:
import torch
import torch.nn as nn

# Simplified illustration of embedding composition in XLNet forward pass

vocab_size = 10  # cluster_num + 2
block_size = 4   # 1 + 1 + cg_depth
n_embd = 32
n_class = 3

# Embedding layers
tok_emb = nn.Embedding(vocab_size, n_embd)
query_emb = nn.Parameter(torch.zeros(1, 1, n_embd))
pos_emb = nn.Parameter(torch.zeros(1, block_size, n_embd))
class_emb = nn.Embedding(n_class, n_embd)

# Example input: batch of 2, sequence length 3 (query tokens, excluding last)
idx = torch.tensor([[9, 3, 5],    # [start_id, root_cluster, hop1_cluster]
                     [9, 7, 1]])
labels = torch.tensor([0, 2])     # node class labels
b, t = idx.size()

# Content stream: token embeddings + position embeddings
token_embeddings = tok_emb(idx)
x = token_embeddings + pos_emb[:, :t]

# Query stream: learnable query + shifted position + class conditioning
class_embeddings = class_emb(labels).unsqueeze(1).expand(-1, t, -1)
q = (query_emb + pos_emb[:, 1:(t+1)] + class_embeddings).expand_as(x)

print("Content stream (x) shape:", x.shape)  # [batch, seq_len, n_embd]
print("Query stream (q) shape:  ", q.shape)   # [batch, seq_len, n_embd]
print()
print("Content stream = tok_emb(tokens) + pos_emb[:t]")
print("Query stream   = query_emb + pos_emb[1:t+1] + class_emb(label)")
print()
print("Note: query stream position is shifted by 1 — it predicts the NEXT token")
print("Note: class embedding injects label information into every query position")

## 4. Training Procedure

Training is implemented in `CGT/generator/gpt/trainer.py` and orchestrated by `CGT/generator/gpt/gpt.py`.

**Data split:** CGT trains on **train + val** nodes combined (`target_ids = ids["train"] + ids["val"]`). This maximises the amount of data the generative model sees. The test split is held out entirely — synthetic data is only generated for train and val nodes.

**Loss:** Standard cross-entropy on next-token prediction. Each short sequence of length `block_size` produces `block_size - 1` predictions (the query at position $t$ predicts the token at position $t+1$). The loss is averaged over all positions and all short sequences in the batch.

**Optimizer:** AdamW ($\beta_1=0.9$, $\beta_2=0.95$) with selective weight decay:
- **Decayed:** weights of `nn.Linear` layers
- **Not decayed:** biases, `nn.LayerNorm` parameters, `nn.Embedding` weights, position embeddings, query embeddings

This follows standard transformer training practice — regularising the large weight matrices while leaving scale/shift parameters and embeddings unregularised.

**Learning rate schedule:**
1. **Linear warmup** over the first epoch (measured in tokens processed)
2. **Cosine decay** from peak LR down to 10% of the peak over remaining training

**Gradient clipping:** Global norm clipped to 1.0 to prevent gradient explosions, which is important given the relatively small model and potentially noisy computation graph inputs.

**Checkpointing:** The model checkpoint is saved at the end of training (no early stopping by default, though `gpt_early_stopping` is available).

In [ ]:
import math
import numpy as np

# Visualise the learning rate schedule used by CGT

def compute_lr_schedule(num_epochs, tokens_per_epoch, base_lr):
    """Replicate the LR schedule from trainer.py."""
    warmup_tokens = tokens_per_epoch  # 1 epoch warmup
    final_tokens = tokens_per_epoch * num_epochs
    min_lr_ratio = 3e-5 / base_lr

    lrs = []
    tokens = 0
    steps_per_epoch = 50  # simulated
    for epoch in range(num_epochs):
        for step in range(steps_per_epoch):
            tokens += tokens_per_epoch // steps_per_epoch
            if tokens < warmup_tokens:
                lr_mult = float(tokens) / float(max(1, warmup_tokens))
            else:
                progress = float(tokens - warmup_tokens) / float(max(1, final_tokens - warmup_tokens))
                lr_mult = max(min_lr_ratio, 0.5 * (1.0 + math.cos(math.pi * progress)))
            lrs.append(base_lr * lr_mult)
    return lrs

# Default hyperparameters
base_lr = 0.003 / 8  # gpt_lr default
num_epochs = 50
tokens_per_epoch = 4 * 25 * 5000  # block_size * short_seq_num * num_nodes (example)

lrs = compute_lr_schedule(num_epochs, tokens_per_epoch, base_lr)

# Print key points
total_steps = len(lrs)
print(f"LR schedule: {num_epochs} epochs, base_lr={base_lr:.6f}")
print(f"  Warmup: 1 epoch ({total_steps // num_epochs} steps)")
print(f"  Peak LR:  {max(lrs):.6f}")
print(f"  Final LR: {lrs[-1]:.6f} ({lrs[-1]/max(lrs)*100:.1f}% of peak)")
print()

# Show LR at key epochs
for e in [0, 1, 5, 10, 25, 40, 49]:
    step = (e + 1) * (total_steps // num_epochs) - 1
    step = min(step, len(lrs) - 1)
    print(f"  Epoch {e:2d}: lr = {lrs[step]:.6f}")

## 5. Autoregressive Generation

After training, CGT generates synthetic computation graphs by autoregressively sampling cluster IDs from the learned distribution (implemented in `CGT/generator/gpt/utils.py`).

**Generation procedure:**

1. Initialise each node's sequence with `[start_id]` and its ground-truth label
2. For each step $s$ in $0, \dots, \text{cg\_depth}$:
   - Forward pass: compute logits from the current sequence
   - Scale logits by temperature: $\text{logits} / T$
   - Sample next token from $\text{softmax}(\text{logits})$ via multinomial sampling
   - Append sampled token to sequence
   - **Expand batch** by repeating each sequence `cg_fanout` times (fan out to the next level of the tree)
3. Concatenate all generated tokens into the full computation graph sequence

The batch expansion at each step is the key mechanism: after generating the root token, the sequence is duplicated `cg_fanout` times, and each copy independently samples its next token. This mirrors the tree structure — each parent node fans out to `cg_fanout` children.

**Temperature** controls the sharpness of the sampling distribution. $T=1.0$ (default) samples from the model's learned distribution. Lower temperatures produce more deterministic (but less diverse) outputs; higher temperatures increase diversity at the cost of coherence.

In [ ]:
import torch
import torch.nn.functional as F

# Simplified illustration of the autoregressive generation loop
# (See CGT/generator/gpt/utils.py for the actual implementation)

torch.manual_seed(42)

cg_depth = 2
cg_fanout = 3
cluster_num = 8
start_id = cluster_num + 1
vocab_size = cluster_num + 1  # head predicts vocab_size - 1 (no start token)
temperature = 1.0

n_nodes = 2  # generate for 2 nodes
labels = torch.tensor([0, 1])

# Start with start tokens
x = torch.full((n_nodes, 1), start_id, dtype=torch.long)

print(f"Generating computation graphs for {n_nodes} nodes")
print(f"  cg_depth={cg_depth}, cg_fanout={cg_fanout}, temperature={temperature}\n")

generated_ids = []
current_labels = labels.clone()

for step in range(cg_depth + 1):
    batch_size = x.size(0)

    # In practice: logits, _ = model(x, current_labels)
    # Here we simulate with random logits
    logits = torch.randn(batch_size, vocab_size)
    logits = logits / temperature

    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    generated_ids.append(next_token)

    # Append to sequence and expand batch for next level
    x = torch.cat((x, next_token), dim=-1)
    x = x.repeat_interleave(cg_fanout, dim=0)
    current_labels = current_labels.repeat_interleave(cg_fanout)

    print(f"  Step {step}: batch_size {batch_size} → sampled {batch_size} tokens → expanded to {x.size(0)}")

# Reassemble into per-node sequences
print(f"\nFinal batch size: {x.size(0)} (= {n_nodes} nodes x {cg_fanout}^{cg_depth+1} expansions)")

# The complete_ids are assembled by concatenating generated tokens at each level
# For node 0: [start, root_token, fanout tokens at hop 1, fanout^2 tokens at hop 2]
complete = torch.full((n_nodes, 1), start_id, dtype=torch.long)
for step in range(cg_depth + 1):
    tokens_at_step = generated_ids[step].view(n_nodes, cg_fanout ** step)
    complete = torch.cat((complete, tokens_at_step), dim=1)

seq_len = complete.size(1)
print(f"\nComplete sequence per node: {seq_len} tokens")
print(f"  = 1 (start) + 1 (root) + {cg_fanout} (hop1) + {cg_fanout**2} (hop2)")
print(f"\nNode 0 sequence: {complete[0].tolist()}")
print(f"Node 1 sequence: {complete[1].tolist()}")

## 6. Output Format and Reconstruction

After training and generation, `train_and_generate()` saves a `.pt` checkpoint containing everything needed to construct synthetic datasets downstream.

**Saved contents** (see `CGT/train.py`):

| Key | Shape | Description |
|---|---|---|
| `gen_train_ids` | `[n_train, seq_len]` | Generated cluster ID sequences for training nodes |
| `gen_val_ids` | `[n_val, seq_len]` | Generated cluster ID sequences for validation nodes |
| `train_labels` | `[n_train]` | Original labels for training nodes |
| `val_labels` | `[n_val]` | Original labels for validation nodes |
| `cluster_centers` | `[cluster_num+1, feat_dim]` | Cluster centre vectors (+ zero vector for empty_id) |
| `ids` | dict | Original train/val/test node ID lists |
| `feat_size` | int | Original feature dimension |
| `label_size` | int | Number of classes |
| `cg_depth` | int | Computation graph depth |
| `cg_fanout` | int | Computation graph fanout |
| `noise_num` | int | Number of noise edges added |
| `self_connection` | bool | Whether self-loops are added |

**Save path:** `datasets/synthetic/cgt/{dataset}/{task}/{variant}/{variant}_t{trial_id}.pt` where variant = `{dataset}_e{epochs}_k{cluster_num}_c{cluster_size}_d{depth}_f{fanout}`.

The variant subdirectory groups all per-trial `.pt` files for a given hyperparameter configuration, keeping different configs (e.g. varying `cluster_size`) cleanly separated.

The `trial_id` suffix (0-9) identifies which GADBench mask column was used for the train/val/test split. Training all 10 trials produces 10 `.pt` files that the evaluation benchmark can load one per trial, matching the split-varying behaviour of the original-data baseline.

### Reconstruction via QuantizedDataset

The `QuantizedDataset` class (`CGT/generator/gpt/dataset.py`) converts generated sequences back into graph data for downstream GNN evaluation:

1. **Features:** Each cluster ID in the generated sequence is mapped to its cluster centre vector. The features for one synthetic node are the concatenated cluster centres for all tokens in its computation graph sequence.

2. **Adjacency:** A **duplicate-encoded adjacency matrix** is constructed — a fixed tree structure determined entirely by `cg_depth` and `cg_fanout`. Every synthetic node has the same topology: a rooted tree where each non-leaf node connects to exactly `cg_fanout` children. If `self_connection=True`, self-loops are added to all nodes.

The duplicate-encoded adjacency is a deliberate design choice: since the generated sequences encode neighbourhood *features* but not explicit *edges*, CGT uses a fixed graph structure that matches the computation graph layout. This enables uniform batching — every synthetic node produces an adjacency matrix of the same shape — at the cost of not capturing the original graph's specific connectivity patterns.

## 7. Key Design Decisions

### Why computation graphs?
Operating on the full adjacency matrix is infeasible for large graphs. Computation graphs decompose the problem into local, fixed-size subgraphs that can be batched and processed independently. This is the same locality assumption that underpins GNN message passing — if a 2-layer GNN only aggregates 2-hop neighbourhoods, then a computation graph of depth 2 captures exactly the information the GNN will use.

### Why quantize features instead of modelling them continuously?
Discrete sequence modelling with cross-entropy loss is well-understood and stable. Modelling continuous features autoregressively would require choosing a likelihood model (e.g., Gaussian mixture) and managing numerical stability. K-means quantization sidesteps this by converting the problem to classification over a fixed vocabulary. The trade-off is a lossy representation bounded by the number of clusters.

### Why train on train+val together?
The generative model benefits from seeing as much data as possible. Since CGT does not use validation loss for model selection (it trains for a fixed number of epochs), there is no methodological issue with including validation nodes in the training set. The test split remains held out to preserve evaluation integrity.

### Why XLNet over standard GPT?
XLNet's query stream allows the model to incorporate richer context when making predictions. In standard GPT, the representation at position $t$ is built from tokens $0, \dots, t$. In XLNet, the query stream at position $t$ attends to the content stream (which has seen tokens $0, \dots, t$) but uses a separate parameterisation. This provides additional modelling capacity without breaking the autoregressive property. The original CGT paper (Chen et al., 2023) found this beneficial for capturing complex feature dependencies in computation graphs.

### Why duplicate-encoded adjacency?
CGT generates features, not edges. Rather than attempting to also generate topology (which would require a second model or a joint formulation), it uses a fixed tree structure that matches the computation graph layout. This means every synthetic node has identical connectivity, which is a significant simplification compared to the original graph. However, it ensures that the GNN operating on synthetic data uses the same aggregation depth and fanout as the computation graphs the features were trained on, maintaining consistency between generation and evaluation.

## 8. Usage

```bash
# Train a single trial (num_trials=1 ⇒ only trial_id=0 is run)
bash scripts/train/train_cgt.sh reddit 50 512 1 128 2 5 1
# Args: [dataset] [gpt_epochs] [cluster_num] [cluster_size] [gpt_batch_size] [cg_depth] [cg_fanout] [num_trials] [task]

# Train all 10 trials (one per GADBench mask column)
bash scripts/train/train_cgt.sh reddit 50 512 1 128 2 5 10

# Or directly (single trial — pick the split via --trial_id)
python CGT/train.py \
    --dataset reddit \
    --data_dir datasets/original \
    --gpt_epochs 50 \
    --cluster_num 512 \
    --cluster_size 1 \
    --gpt_batch_size 128 \
    --cg_depth 2 \
    --cg_fanout 5 \
    --trial_id 0
```

The shell script is idempotent: trials whose output `.pt` already exists are skipped, so re-submitting after a SLURM timeout resumes where it left off.

Training reads the precomputed clustering partition from `cache/clustering/` (see Section 1). The matching cache entry for `(dataset, task, trial, cluster_size, cluster_num)` must already be materialised by `scripts/cluster/precompute_clusters.py`, otherwise training fails loud. `train_cgt.sh` runs from the project root, so the default `--cache_root cache/clustering` resolves without any extra flag.

**Key hyperparameters:**

| Parameter | Default | Description |
|---|---|---|
| `--gpt_epochs` | 50 | Training epochs |
| `--cluster_num` | 512 | Number of k-means clusters (vocab size - 2); selects the cache leaf |
| `--cluster_size` | 1 | Minimum cluster size (k-anonymity floor); selects the cache leaf |
| `--cache_root` | cache/clustering | Root of the precomputed clustering cache (relative to project root unless absolute) |
| `--gpt_batch_size` | 128 | Batch size (number of nodes per batch) |
| `--cg_depth` | 2 | Hops in computation graph |
| `--cg_fanout` | 5 | Neighbours sampled per hop |
| `--trial_id` | 0 | GADBench mask column for train/val/test split (0-9); selects the cache `t<trial>` (hidden_labels only) |
| `--task` | hidden_labels | Pipeline task (determines output subdirectory and cache layout) |
| `--gpt_layers` | 3 | Transformer layers |
| `--gpt_heads` | 12 | Attention heads |
| `--gpt_hidden_dim` | 64 | Hidden dim per head (total = hidden_dim * heads) |
| `--gpt_lr` | 0.000375 | Learning rate (0.003 / 8) |
| `--gpt_dropout` | 0.2 | Dropout rate |
| `--gpt_softmax_temperature` | 1.0 | Sampling temperature during generation |
| `--cluster_sample_num` | 5000 | (Inert for the partition — baked into the cache; still used in the variant dir name) |
| `--noise_num` | 0 | Random noise neighbours to add per hop |
| `--self_connection` | True | Add self-loops in duplicate adjacency |

## References

- Chen, Y., Zhang, Y., Bian, T., Chen, H., Karypis, G. & Li, J. (2023). *Demystifying Graph Condensation with Computation Graphs.* (CGT)
- Yang, Z., Dai, Z., Yang, Y., Carbonell, J., Salakhutdinov, R. & Le, Q. V. (2019). *XLNet: Generalized Autoregressive Pretraining for Language Understanding.* NeurIPS.
- Karpathy, A. (2020). *minGPT.* https://github.com/karpathy/minGPT — CGT's transformer implementation is adapted from this codebase.
- Loshchilov, I. & Hutter, F. (2019). *Decoupled Weight Decay Regularization.* ICLR. (AdamW optimizer)